# Local PDF Question-Answering System

## Overview
This project implements a local Question-Answering (QA) pipeline using a Retrieval-Augmented Generation (RAG) architecture. It enables users to query uploaded PDF documents entirely offline.

## Technical Architecture

### Models
*   **Generative QA Model**: `google/flan-t5-large`  
    A sequence-to-sequence (Seq2Seq) model that generates natural, human-like answers based on retrieved context.
*   **Embedding Model**: `sentence-transformers/all-MiniLM-L6-v2`  
    A lightweight model that converts text into 384-dimensional vectors to capture semantic meaning for retrieval.

### Pipeline Components
1.  **Dynamic Chunking**: The system automatically adjusts chunk sizes based on the document's total token count (using `tiktoken`), ensuring that even short documents are properly segmented for retrieval.
2.  **In-Memory Vector Search**: Document chunks are stored as PyTorch Tensors in RAM. The system uses **Cosine Similarity** to rank chunks against the user's query.
3.  **RAG Logic**: The top-ranked chunk is passed to the generative model with a specific system instruction prompt to ensure accuracy and professionalism.

### Retrieval Process
1.  **Vectorization**: Text chunks are converted into semantic embeddings.
2.  **Similarity Search**: Cosine similarity is computed between the query vector and all chunk vectors.
3.  **Generation**: The most relevant chunk (Top-3) is used as context for the FLAN-T5 model to generate the final answer.

---
*Note: Set the Colab runtime to GPU (Runtime > Change runtime type > T4 GPU) for optimal performance.*

---
# Block 1: Setup and Installation

Install required libraries and verify the runtime environment.

In [1]:
# Install all required libraries
!pip install -q pymupdf transformers sentencepiece rouge-score tiktoken accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 62.0 MB/s eta 0:00:00


In [21]:
import fitz
import textwrap
import tiktoken
import torch
from rouge_score import rouge_scorer
from google.colab import files

print("Libraries imported successfully")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

Libraries imported successfully
GPU available: True
Device: Tesla T4
Libraries imported successfully
GPU available: True
Device: Tesla T4


In [3]:
USE_SAMPLE = False   # Set to False to upload your own PDF

if USE_SAMPLE:
    !wget -q -O sample.pdf https://arxiv.org/pdf/1706.03762  # Attention Is All You Need
    PDF_PATH = "sample.pdf"
    print("Downloaded sample PDF: Attention Is All You Need")
else:
    print("Please upload your PDF file:")
    uploaded = files.upload()
    if not uploaded:
        print("No file uploaded. Please run the cell again to upload or set USE_SAMPLE = True.")
    else:
        PDF_PATH = list(uploaded.keys())[0]
        print(f"\nSuccessfully uploaded: {PDF_PATH}")

Please upload your PDF file:


Saving Test PDF (Hard).pdf to Test PDF (Hard).pdf

Successfully uploaded: Test PDF (Hard).pdf
Please upload your PDF file:


---
# Block 2: PDF Text Extraction
Use PyMuPDF (`fitz`) to extract page-level text, then clean and normalize it.

Discussion: PDFs store content by rendering order, not always reading order. Extraction strategy matters.

In [4]:
def extract_text_from_pdf(pdf_path: str) -> dict:
    """
    Extract text from each page of a PDF.
    Returns a dict with metadata and a list of page texts.
    """
    doc = fitz.open(pdf_path)

    result = {
        "num_pages": len(doc),
        "pages": []
    }

    for page_num, page in enumerate(doc):
        text = page.get_text("text")  # plain text mode
        result["pages"].append({
            "page": page_num + 1,
            "text": text
        })

    doc.close()
    return result


pdf_data = extract_text_from_pdf(PDF_PATH)
full_text_raw = "\n".join(p['text'] for p in pdf_data['pages'])

print(f"Pages : {pdf_data['num_pages']}")
print("\n--- First 500 chars of page 1 (raw) ---")
print(pdf_data['pages'][0]['text'][:500])
print(f"\nTotal characters: {len(full_text_raw):,}")
print(f"Total words: {len(full_text_raw.split()):,}")

Pages : 3

--- First 500 chars of page 1 (raw) ---
Bangladesh(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&
&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&
&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%
%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%
%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$
$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((
((((((_. is a beautiful country in South Asia, kn

Total characters: 4,819
Total words: 404


In [5]:
import re
import string

def clean_text(text: str) -> str:
    """
    Aggressive cleaning to remove PDF artifacts:
    - Remove specific noise characters like $, %, &, (, ), _
    - Remove non-printable characters and zero-width spaces
    - Collapse multiple newlines and horizontal spaces
    """
    # Remove the noise characters identified in the PDF
    text = re.sub(r'[\$%&()_]', '', text)

    # Remove zero-width characters and other non-printable artifacts
    text = "".join(filter(lambda x: x in string.printable, text))

    # Collapse excessive whitespace
    text = re.sub(r'[ \t]+', ' ', text) # Horizontal space
    text = re.sub(r'\n\s*\n', '\n\n', text) # Vertical space
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

# Re-process the text
full_text_raw = "\n".join(p['text'] for p in pdf_data['pages'])
full_text = clean_text(full_text_raw)

print("--- Cleaned text (first 500 chars) ---")
print(full_text[:500])
print(f"\nTotal characters: {len(full_text):,}")

--- Cleaned text (first 500 chars) ---
Bangladesh

. is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and 
vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political 
center but also the economic and cultural heart of the nation. It is one of the most densely 
populated cities in the world and plays a major role in the countrys development. 

One of the most important natural features of Bangladesh is the 

. 
Padma River, which is often referred to as the lif

Total characters: 2,353


In [6]:
# Count tokens before selecting a summarization strategy
enc = tiktoken.get_encoding("cl100k_base")  # #https://tiktokenizer.vercel.app/
total_tokens = len(enc.encode(full_text))

print(f"Total tokens: {total_tokens:,}")
print()
print("Model context limits:")
print(f"  BERT          : 512 tokens  -> fits {512/total_tokens*100:.1f}% of the document")
print(f"  BART-large    : 1024 tokens -> fits {1024/total_tokens*100:.1f}% of the document")
print(f"  GPT-3.5       : 4096 tokens -> fits {4096/total_tokens*100:.1f}% of the document")
print(f"  GPT-4 / Claude: 128K tokens -> fits {min(128000/total_tokens*100, 100):.1f}% of the document")
print()
print("Chunking is required for BART on long documents.")

Total tokens: 494

Model context limits:
  BERT          : 512 tokens  -> fits 103.6% of the document
  BART-large    : 1024 tokens -> fits 207.3% of the document
  GPT-3.5       : 4096 tokens -> fits 829.1% of the document
  GPT-4 / Claude: 128K tokens -> fits 100.0% of the document

Chunking is required for BART on long documents.


In [7]:
def chunk_text_dynamic(text, overlap_percent=0.1):
    """
    Splits text into chunks using tiktoken.
    Dynamically adjusts chunk size based on document length to ensure multiple chunks.
    """
    tokens = enc.encode(text)
    total_tokens = len(tokens)

    # Dynamic chunk size: aim for at least 4-6 chunks, but keep within reasonable bounds (100-500 tokens)
    if total_tokens < 500:
        base_chunk_size = max(100, total_tokens // 4)
    else:
        base_chunk_size = 500

    overlap = int(base_chunk_size * overlap_percent)

    chunks = []
    for i in range(0, total_tokens, base_chunk_size - overlap):
        chunk_tokens = tokens[i : i + base_chunk_size]
        chunk_text = enc.decode(chunk_tokens)
        chunks.append(chunk_text)
        if i + base_chunk_size >= total_tokens:
            break

    return chunks, base_chunk_size, overlap


In [8]:
from transformers import AutoModel, AutoTokenizer
import torch

# Define device for hardware acceleration
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize the embedding model (MiniLM as specified in the architecture)
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_model_name)
embed_model = AutoModel.from_pretrained(embed_model_name).to(device)

def compute_embeddings(text_list):
    """
    Converts a list of strings into a tensor of semantic embeddings.
    """
    inputs = embed_tokenizer(text_list, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        model_output = embed_model(**inputs)
        # Perform pooling (mean pooling) to get sentence embeddings
        embeddings = model_output.last_hidden_state.mean(dim=1)
        # Normalize embeddings for cosine similarity
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    return embeddings

print(f"Embedding model '{embed_model_name}' loaded successfully on {device}.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model 'sentence-transformers/all-MiniLM-L6-v2' loaded successfully on cuda.


In [9]:
# Re-apply dynamic chunking and re-calculate embeddings
chunks, current_size, current_overlap = chunk_text_dynamic(full_text)
print(f"Document Tokens: {total_tokens}")
print(f"Dynamic Chunk Size: {current_size}")
print(f"Total chunks created: {len(chunks)}")

Document Tokens: 494
Dynamic Chunk Size: 123
Total chunks created: 5


In [10]:
# Refresh embeddings with the newly created dynamic chunks
chunk_embeddings = compute_embeddings(chunks)
print(f"Generated embeddings for {len(chunks)} chunks.")

Generated embeddings for 5 chunks.


In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Using FLAN-T5 Large for superior generative QA performance locally
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def answer_question_local(question, context):
    """
    Performs generative QA using FLAN-T5 with a detailed system-style prompt.
    """
    system_prompt = (
        "You are a helpful and accurate assistant. Use the provided context to answer the user's question. "
        "If the answer is not in the context, say you do not know. Keep the answer concise and professional."
    )

    prompt = f"Instruction: {system_prompt}\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            inputs["input_ids"],
            max_new_tokens=200,
            num_beams=4,
            early_stopping=True
        )

    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return answer.strip()

print(f"Generative QA Model '{model_name}' loaded successfully with enhanced prompt logic.")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generative QA Model 'google/flan-t5-large' loaded successfully with enhanced prompt logic.


# Block 3.5: Local Embedding and Retrieval (RAG)
Instead of searching every chunk, we will embed them and use cosine similarity to find the most relevant ones.

In [12]:
def retrieve_relevant_chunks(question, chunk_embeddings, chunks, top_k=3):
    # Embed the user question
    question_embedding = compute_embeddings([question])
    # Compute cosine similarity
    similarities = torch.matmul(question_embedding, chunk_embeddings.T).squeeze(0)
    # Get top-k indices based on similarity scores
    top_indices = torch.topk(similarities, k=min(top_k, len(chunks))).indices.tolist()
    return [chunks[i] for i in top_indices]

# Updated RAG-style ask function to use Top-3 chunks for better context coverage
def ask_pdf_rag(question, chunks, chunk_embeddings):
    relevant_chunks = retrieve_relevant_chunks(question, chunk_embeddings, chunks, top_k=3)

    if not relevant_chunks:
        return "No relevant content found.", []

    # Combine the top 3 chunks into a single context block
    combined_context = "\n\n---\n\n".join(relevant_chunks)
    answer = answer_question_local(question, combined_context)

    # Check if the model failed to find an answer
    if not answer.strip() or len(answer) < 2 or "do not know" in answer.lower():
        return "The model could not find a specific answer in the provided context.", relevant_chunks

    return answer, relevant_chunks

### Ask a question
You can now query the PDF content directly.

In [14]:
user_question = "What is the capital city of Bangladesh?"
answer, retrieved_contexts = ask_pdf_rag(user_question, chunks, chunk_embeddings)

print(f"QUESTION: {user_question}")
print(f"\nANSWER:\n{answer}")

print("\n" + "="*50)
print("RETRIEVED SOURCE CHUNKS:")
for i, ctx in enumerate(retrieved_contexts):
    print(f"\n--- Chunk {i+1} ---")
    print(ctx[:400] + "...")

QUESTION: What is the capital city of Bangladesh?

ANSWER:
DHAKA

RETRIEVED SOURCE CHUNKS:

--- Chunk 1 ---
Bangladesh

. is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and 
vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political 
center but also the economic and cultural heart of the nation. It is one of the most densely 
populated cities in the world and plays a major role in the countrys development. 

One of the most imp...

--- Chunk 2 ---
. 

The official currency of Bangladesh is the Bangladeshi Taka. It is used in all types of financial 
transactions across the country. 

. The 
economy of Bangladesh has been growing steadily, with key sectors including agriculture, 
textiles, and remittances from abroad. 

The national animal of Bangladesh is the Royal Bengal Tiger. This majestic animal represents 
strength, courage, and pride. ...

--- Chunk 3 ---
, also known as Bangla. It is spoken by the 
major

In [15]:
user_question = "Which river is considered the lifeline of Bangladesh?"
answer, retrieved_contexts = ask_pdf_rag(user_question, chunks, chunk_embeddings)

print(f"QUESTION: {user_question}")
print(f"\nANSWER:\n{answer}")

print("\n" + "="*50)
print("RETRIEVED SOURCE CHUNKS:")
for i, ctx in enumerate(retrieved_contexts):
    print(f"\n--- Chunk {i+1} ---")
    print(ctx[:400] + "...")

QUESTION: Which river is considered the lifeline of Bangladesh?

ANSWER:
Padma River

RETRIEVED SOURCE CHUNKS:

--- Chunk 1 ---
Bangladesh

. is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and 
vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political 
center but also the economic and cultural heart of the nation. It is one of the most densely 
populated cities in the world and plays a major role in the countrys development. 

One of the most imp...

--- Chunk 2 ---
 of the country. This river is crucial for 
transportation, agriculture, 

fishing, and daily life. Many people depend on it for their livelihoods, and it supports the fertile 
land that makes Bangladesh suitable for farming. 

Bangladesh gained its independence in 1971. This historic event is remembered every year as 
Independence Day and is a source of great pride 
for the people of Bangladesh. ...

--- Chunk 3 ---
, also known as Bangla. It is 

In [16]:
user_question = "In which year did Bangladesh gain independence?"
answer, retrieved_contexts = ask_pdf_rag(user_question, chunks, chunk_embeddings)

print(f"QUESTION: {user_question}")
print(f"\nANSWER:\n{answer}")

print("\n" + "="*50)
print("RETRIEVED SOURCE CHUNKS:")
for i, ctx in enumerate(retrieved_contexts):
    print(f"\n--- Chunk {i+1} ---")
    print(ctx[:400] + "...")

QUESTION: In which year did Bangladesh gain independence?

ANSWER:
1971

RETRIEVED SOURCE CHUNKS:

--- Chunk 1 ---
 of the country. This river is crucial for 
transportation, agriculture, 

fishing, and daily life. Many people depend on it for their livelihoods, and it supports the fertile 
land that makes Bangladesh suitable for farming. 

Bangladesh gained its independence in 1971. This historic event is remembered every year as 
Independence Day and is a source of great pride 
for the people of Bangladesh. ...

--- Chunk 2 ---
, also known as Bangla. It is spoken by the 
majority of the population and is deeply connected to the countrys culture, literature, and 
traditions. The importance of the language is highlighted by International Mother Language Day, 
which originated from the language movement in Bangladesh. 

. 

Bangladesh is also famous for the Sundarbans, the largest mangrove forest in the world. This 
forest...

--- Chunk 3 ---
. 

The official currency of Bangladesh is 

In [17]:
user_question = "What is the official language of Bangladesh?"
answer, retrieved_contexts = ask_pdf_rag(user_question, chunks, chunk_embeddings)

print(f"QUESTION: {user_question}")
print(f"\nANSWER:\n{answer}")

print("\n" + "="*50)
print("RETRIEVED SOURCE CHUNKS:")
for i, ctx in enumerate(retrieved_contexts):
    print(f"\n--- Chunk {i+1} ---")
    print(ctx[:400] + "...")

QUESTION: What is the official language of Bangladesh?

ANSWER:
Bengali

RETRIEVED SOURCE CHUNKS:

--- Chunk 1 ---
, also known as Bangla. It is spoken by the 
majority of the population and is deeply connected to the countrys culture, literature, and 
traditions. The importance of the language is highlighted by International Mother Language Day, 
which originated from the language movement in Bangladesh. 

. 

Bangladesh is also famous for the Sundarbans, the largest mangrove forest in the world. This 
forest...

--- Chunk 2 ---
 of the country. This river is crucial for 
transportation, agriculture, 

fishing, and daily life. Many people depend on it for their livelihoods, and it supports the fertile 
land that makes Bangladesh suitable for farming. 

Bangladesh gained its independence in 1971. This historic event is remembered every year as 
Independence Day and is a source of great pride 
for the people of Bangladesh. ...

--- Chunk 3 ---
. 

The official currency of Bangladesh is 

In [18]:
user_question = "What is the name of the world’s largest mangrove forest located in Bangladesh?"
answer, retrieved_contexts = ask_pdf_rag(user_question, chunks, chunk_embeddings)

print(f"QUESTION: {user_question}")
print(f"\nANSWER:\n{answer}")

print("\n" + "="*50)
print("RETRIEVED SOURCE CHUNKS:")
for i, ctx in enumerate(retrieved_contexts):
    print(f"\n--- Chunk {i+1} ---")
    print(ctx[:400] + "...")

QUESTION: What is the name of the world’s largest mangrove forest located in Bangladesh?

ANSWER:
Sundarbans

RETRIEVED SOURCE CHUNKS:

--- Chunk 1 ---
, also known as Bangla. It is spoken by the 
majority of the population and is deeply connected to the countrys culture, literature, and 
traditions. The importance of the language is highlighted by International Mother Language Day, 
which originated from the language movement in Bangladesh. 

. 

Bangladesh is also famous for the Sundarbans, the largest mangrove forest in the world. This 
forest...

--- Chunk 2 ---
Bangladesh

. is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and 
vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political 
center but also the economic and cultural heart of the nation. It is one of the most densely 
populated cities in the world and plays a major role in the countrys development. 

One of the most imp...

--- Chunk 3 ---
 Bangl

In [19]:
user_question = "Which currency is used in Bangladesh?"
answer, retrieved_contexts = ask_pdf_rag(user_question, chunks, chunk_embeddings)

print(f"QUESTION: {user_question}")
print(f"\nANSWER:\n{answer}")

print("\n" + "="*50)
print("RETRIEVED SOURCE CHUNKS:")
for i, ctx in enumerate(retrieved_contexts):
    print(f"\n--- Chunk {i+1} ---")
    print(ctx[:400] + "...")

QUESTION: Which currency is used in Bangladesh?

ANSWER:
Bangladeshi Taka

RETRIEVED SOURCE CHUNKS:

--- Chunk 1 ---
. 

The official currency of Bangladesh is the Bangladeshi Taka. It is used in all types of financial 
transactions across the country. 

. The 
economy of Bangladesh has been growing steadily, with key sectors including agriculture, 
textiles, and remittances from abroad. 

The national animal of Bangladesh is the Royal Bengal Tiger. This majestic animal represents 
strength, courage, and pride. ...

--- Chunk 2 ---
, also known as Bangla. It is spoken by the 
majority of the population and is deeply connected to the countrys culture, literature, and 
traditions. The importance of the language is highlighted by International Mother Language Day, 
which originated from the language movement in Bangladesh. 

. 

Bangladesh is also famous for the Sundarbans, the largest mangrove forest in the world. This 
forest...

--- Chunk 3 ---
 of the country. This river is crucial fo

In [20]:
user_question = "What is the national animal of Bangladesh?"
answer, retrieved_contexts = ask_pdf_rag(user_question, chunks, chunk_embeddings)

print(f"QUESTION: {user_question}")
print(f"\nANSWER:\n{answer}")

print("\n" + "="*50)
print("RETRIEVED SOURCE CHUNKS:")
for i, ctx in enumerate(retrieved_contexts):
    print(f"\n--- Chunk {i+1} ---")
    print(ctx[:400] + "...")

QUESTION: What is the national animal of Bangladesh?

ANSWER:
Royal Bengal Tiger

RETRIEVED SOURCE CHUNKS:

--- Chunk 1 ---
. 

The official currency of Bangladesh is the Bangladeshi Taka. It is used in all types of financial 
transactions across the country. 

. The 
economy of Bangladesh has been growing steadily, with key sectors including agriculture, 
textiles, and remittances from abroad. 

The national animal of Bangladesh is the Royal Bengal Tiger. This majestic animal represents 
strength, courage, and pride. ...

--- Chunk 2 ---
, also known as Bangla. It is spoken by the 
majority of the population and is deeply connected to the countrys culture, literature, and 
traditions. The importance of the language is highlighted by International Mother Language Day, 
which originated from the language movement in Bangladesh. 

. 

Bangladesh is also famous for the Sundarbans, the largest mangrove forest in the world. This 
forest...

--- Chunk 3 ---
 of the country. This river is cru